# Notebook 03 — Preprocessing & Feature Engineering

This notebook prepares data for predicting annual rent in JVC.  
We will:
- Load cleaned data from SQLite.
- Define the target (`price_yearly_aed`) and features.
- Create derived features (area per bedroom, bathrooms per bedroom, etc.).
- Encode categorical features.
- Save a CSV for ML and update the database for the dashboard.

## Imports

In [6]:
import pandas as pd
import sqlite3
import numpy as np
import os


## Load data

We load the cleaned data that we saved earlier in the SQLite DB

In [7]:
DB_PATH = os.path.join("data", "database.db")

conn = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM jvc_apartments", conn)
conn.close()

print("Loaded shape:", df.shape)

df.head()


Loaded shape: (951, 14)


,title,price,property_type,frequency,bedrooms,bathrooms,area,location,url,price_clean,price_yearly_aed,bedrooms_clean,bathrooms_clean,area_clean
0,1BR Apartment for Rent | Balcony & Pool | JVC,"74,000",apartment,yearly,1,2,905 sqft,"AAA Residence, JVC District 13, Jumeirah Villa...",https://www.bayut.com/property/details-9398172...,74000,74000,1.0,2.0,905.0
1,1 B/R with Balcony | Pool & Gym | JVC,"69,000",apartment,yearly,1,1,883 sqft,"Emerald Tower, JVC District 18, Jumeirah Villa...",https://www.bayut.com/property/details-4864285...,69000,69000,1.0,1.0,883.0
2,"Binghatti Phoenix, Jumeirah Village Circle, Dubai","89,990",apartment,yearly,1,2,826 sqft,"Binghatti Phoenix, JVC District 13, Jumeirah V...",https://www.bayut.com/property/details-1347054...,89990,89990,1.0,2.0,826.0
3,Converted into 2BR | Private Garden | Furnished,"140,000",apartment,yearly,1,2,"1,133 sqft","Signature Livings South, Signature Livings, JV...",https://www.bayut.com/property/details-1330702...,140000,140000,1.0,2.0,1133.0
4,Spacious 1Br | Prime Location | JVC,"75,000",apartment,yearly,1,2,925 sqft,"Reef Residence, JVC District 13, Jumeirah Vill...",https://www.bayut.com/property/details-1366409...,75000,75000,1.0,2.0,925.0


## Target


Before going into the feature engineering, I define the target first. My Target for this current project is the yearly price/rent of the property `price_yearly_aed`

Since we are predicting a number, this is a Regression problem.
_(There are two types of ML tasks, regression and classification, regression is when we predict a number, whereas classification is when we predict a category)_    

!!! Nothing derived from the target field should be fed into the model, like in our case the `price_per_sqft`, this is done to prevent data leakage.

In [8]:
TARGET = "price_yearly_aed"


## Feature Engineering

We organize the features as such:
- Base Numerical
  - `bedrooms_clean`
  - `bathrooms_clean`
  - `area_clean`

- Engineered numerical (What we'll create/handle)
  - `price_per_sqft`  (LEAKAGE — don’t include)
  - `area_per_bedroom`
  - `bathrooms_per_bedroom`
  - `log_area`
  - `log_price`

- Categorical
  - `property_type`
  - `building`

We start by area by bedroom, this tells us wether the property is spacious or crampy.

In [9]:
df["area_per_bedroom"] = df["area_clean"] / df["bedrooms_clean"]


Next, bathroom per bedroom. This gives a sense of luxury, because a 2BR property with 1 bathroom is not like a 2BR property with 2 bathrooms.

In [10]:
df["bathrooms_per_bedroom"] = df["bathrooms_clean"] / df["bedrooms_clean"]


### Log transforms (for skew)

Why logs?
- Some models assume features are linearly related to the prediction target. But in our case, real estate prices are almost never linear, they are skewed.
- For example, in my dataset, you can see some outliers that create a huge gap between the listings. These rare listings with the high rental price tend to dominate the learning in a way that isn't helpful.

What does log do?

```
  5000  → something like 8.5  
  10000 → 9.2  
  15000 → 9.6  
  200000 → 12.2  
  300000 → 12.6    
```
- compresses huge values
- spread out smaller values
- Balances the data

In [11]:
df["log_area"] = np.log1p(df["area_clean"])
df["log_price"] = np.log1p(df["price_yearly_aed"]) # This is for modeling, not for EDA!!


### Derive building features

_We extract structured features like building, district, and engineered variables from raw fields so the model can learn meaningful patterns from the data, and later we encode and use them as numeric inputs for training machine-learning models rather than feeding raw text, which models can't interpret directly._

In [12]:
# 1) Extract building (everything before "JVC District <number>")
df["building"] = df["location"].str.extract(r"^(.*?),\s*JVC District \d+", expand=False)
df["building"] = df["building"].fillna(df["location"].str.rsplit(",", n=2).str[0].str.strip())
print("Sample buildings:", df["building"].unique()[:5])

# 2️) Extract district (the part that contains "District")
df["district"] = df["location"].str.extract(r"(JVC District \d+)", expand=False).fillna("Unknown")
print("Sample districts:", df["district"].unique()[:5])

# 3️3) Extract community (second-to-last part before city, usually Dubai)
df["community"] = df["location"].str.split(",").str[-2].str.strip().fillna("Unknown")
print("Sample communities:", df["community"].unique()[:5])


Sample buildings: ['AAA Residence' 'Emerald Tower' 'Binghatti Phoenix'
 'Signature Livings South, Signature Livings' 'Reef Residence']
Sample districts: ['JVC District 13' 'JVC District 18' 'JVC District 10' 'JVC District 11'
 'JVC District 14']
Sample communities: ['Jumeirah Village Circle (JVC)']


## Handle categorical features

We have a problem, many unique building values.     
Having a high-cardinality feature creates many problems:
- **Exploding dimensions**: one-hot encoding creates hundreds of dummy columns, creating a sparse dataset. (slower training and more memory usage)
- **Overfitting**: models like tree-based ones can "memorize" rare categories that only appear in the training set instead of learning the general rules.


So we keep the top N building selection (based on listings) and group the rest to an "Other" category.

In [13]:
TOP_N_BUILDINGS = 20

top_buildings = (
    df["building"]
    .value_counts()
    .head(TOP_N_BUILDINGS)
    .index
)

df["building_grouped"] = df["building"].where(
    df["building"].isin(top_buildings),
    "Other"
)


_Quick sanity check_

In [14]:
# General shape
print("Dataset shape:", df.shape)

# Check column names and types
print("\nColumns and data types:\n", df.dtypes)

# First 5 rows
print("\nSample rows:\n", df.head())

# Check for missing values
print("\nMissing values per column:\n", df.isna().sum())


Dataset shape: (951, 22)

Columns and data types:
 title                     object
price                     object
property_type             object
frequency                 object
bedrooms                   int64
bathrooms                  int64
area                      object
location                  object
url                       object
price_clean                int64
price_yearly_aed           int64
bedrooms_clean           float64
bathrooms_clean          float64
area_clean               float64
area_per_bedroom         float64
bathrooms_per_bedroom    float64
log_area                 float64
log_price                float64
building                  object
district                  object
community                 object
building_grouped          object
dtype: object

Sample rows:
                                                title    price property_type  \
0      1BR Apartment for Rent | Balcony & Pool | JVC   74,000     apartment   
1              1 B/R with Balcony | 

In [16]:
cat_features = ["property_type", "building", "building_grouped", "district", "community"]

for col in cat_features:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(10))



--- property_type ---
property_type
apartment    869
villa         66
townhouse     15
penthouse      1
Name: count, dtype: int64

--- building ---
building
Binghatti Phoenix    39
Binghatti Phantom    37
Binghatti Royale     36
SH Living 1          18
Fortunato            17
Pearl House 2        17
Imperial Tower       17
Reef Residence       17
Rose 10              14
Binghatti Crest      14
Name: count, dtype: int64

--- building_grouped ---
building_grouped
Other                607
Binghatti Phoenix     39
Binghatti Phantom     37
Binghatti Royale      36
SH Living 1           18
Imperial Tower        17
Fortunato             17
Reef Residence        17
Pearl House 2         17
Rose 10               14
Name: count, dtype: int64

--- district ---
district
JVC District 13    156
JVC District 10    142
JVC District 11    140
JVC District 15    130
JVC District 14    115
JVC District 12    103
JVC District 18     70
JVC District 17     63
JVC District 16     29
Unknown              3


## Encoding

In [17]:
cat_features = ["property_type", "building_grouped", "district"] # these are the feature we will encode

df_encoded = pd.get_dummies(df, columns=cat_features, drop_first=True) # creates binary columns, and we drop the first



### Prepare X and y

_X is the training data, and y is the target_

In [19]:

# These are the features that existed normally in the dataset
FEATURES = [
    "bedrooms_clean",
    "bathrooms_clean",
    "area_clean",
    "area_per_bedroom",
    "bathrooms_per_bedroom",
    "log_area"
]

# And these are all the columns created when we encoded
ohe_cols = [col for col in df_encoded.columns if any(f"{cat}_" in col for cat in cat_features)]
#for every column in the encoded dataset, we loop through the features list `cat_features` to see of this column
# starts with any feature name, bec when we encoded we appended the feature name to the encoded new column
# like for example: 'property_type_penthouse', 'building_grouped_Binghatti Corner', ...

print(ohe_cols)
FEATURES += ohe_cols


X = df_encoded[FEATURES] # setting training features
y = df_encoded[TARGET]  # TARGET


print("X shape:", X.shape)
print("y shape:", y.shape)




['property_type_penthouse', 'property_type_townhouse', 'property_type_villa', 'building_grouped_Binghatti Corner', 'building_grouped_Binghatti Crest', 'building_grouped_Binghatti Heights', 'building_grouped_Binghatti House', 'building_grouped_Binghatti Phantom', 'building_grouped_Binghatti Phoenix', 'building_grouped_Binghatti Royale', 'building_grouped_Bloom Heights 1, Bloom Heights', 'building_grouped_DAMAC Ghalia', 'building_grouped_Fortunato', 'building_grouped_Imperial Tower', 'building_grouped_Laya Residences', 'building_grouped_Other', 'building_grouped_Pearl House 2', 'building_grouped_Reef Residence', 'building_grouped_Rigel Apartments', 'building_grouped_Rose 10', 'building_grouped_SH Living 1', 'building_grouped_Sydney Villas', 'building_grouped_Westview Garden', 'district_JVC District 11', 'district_JVC District 12', 'district_JVC District 13', 'district_JVC District 14', 'district_JVC District 15', 'district_JVC District 16', 'district_JVC District 17', 'district_JVC Distr

## Saving

So in this part i will save the dataset we've been working on up until now as a csv file because its easier to load to colab or any ML environment. Then i will update the DB, so it will hold proper data to be displayed (Visualized)

In [31]:
# Paths
DATA_FOLDER = "data"
DB_PATH = os.path.join(DATA_FOLDER, "database.db")
CSV_ML_PATH = os.path.join(DATA_FOLDER, "jvc_apartments_ml.csv")

# SAVE CSV FOR ML =====
# df_encoded = dataframe holding all columns including the encoding
# df = dataframe with all the features, but without the encoding (for the DB)

df_encoded.to_csv(CSV_ML_PATH, index=False)
print(f"CSV ML saved → {CSV_ML_PATH}")


#  Sanity check CSV
csv_check = pd.read_csv(CSV_ML_PATH)
print("CSV ML shape:", csv_check.shape)
# print("Columns preview:", csv_check.columns.tolist()[:10])
# print("First 3 rows:\n", csv_check.head(3))


# UPDATE DASHBOARD DB =====
DB_CONN = sqlite3.connect(DB_PATH)

# If exists we replace
df.to_sql("jvc_apartments", DB_CONN, if_exists="replace", index=False)


# # Sanity check DB
conn_check = sqlite3.connect(DB_PATH)
query_check = pd.read_sql("SELECT COUNT(*) AS total_rows FROM jvc_apartments", conn_check)
print("DB row count:", query_check["total_rows"].iloc[0])

# # Verify first 5 rows or the BD
# sample_db = pd.read_sql("SELECT * FROM jvc_apartments LIMIT 5", conn_check)
# print("Sample DB rows:\n", sample_db)

DB_CONN.close()
print(f"Database updated → {DB_PATH}")

print(f"Number of listings match: {csv_check.shape[0]}") if \
    csv_check.shape[0] == query_check["total_rows"].iloc[0] else \
    print("Number of listings don't match")


CSV ML saved → data/jvc_apartments_ml.csv
CSV ML shape: (951, 51)
DB row count: 951
Database updated → data/database.db
Number of listings match: 951


# Conclusion
- Numeric and categorical features are ready for ML.
- Encoded dataset saved as `jvc_apartments_ml.csv`.
- SQLite database `database.db` updated for the dashboard.
- Next notebook: 04 — Model training and evaluation.
